In [1]:
import os
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    roc_curve,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

# --------------------
# Warnings / seed
# --------------------
warnings.filterwarnings("ignore", message="Some weights of the model checkpoint.*")
warnings.filterwarnings("ignore", message="Some weights of .* were not initialized.*")
warnings.filterwarnings("ignore", message="You should probably TRAIN this model.*")
warnings.filterwarnings("ignore", message="You're using a PreTrainedTokenizerFast.*")

BASE_SEED = 42

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# --------------------
# Paths / Config
# --------------------
MODEL_DIR = "model"
REPS_ROOT = "./data"   # rep1..rep10
N_REP = 10

OUT_ROOT = "./dnabert2_by_rep_valauc_valf1_grid001"
os.makedirs(OUT_ROOT, exist_ok=True)

SEQ_COL = "sequence"
LABEL_COL = "label"
MAX_LEN = 256

# --------------------
# Data helpers
# --------------------
def clean_seq(s: str) -> str:
    s = str(s).upper()
    return "".join([c if c in "ACGT" else "N" for c in s])

def load_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if SEQ_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"{path} must contain columns: [{SEQ_COL}, {LABEL_COL}]")

    df[SEQ_COL] = df[SEQ_COL].astype(str).map(clean_seq)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    df = df[df[LABEL_COL].isin([0, 1])].reset_index(drop=True)

    if len(df) == 0:
        raise ValueError(f"{path} is empty after filtering labels.")
    return df

# --------------------
# Metrics helpers
# --------------------
def _softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def _extract_logits(predictions):
    if isinstance(predictions, (tuple, list)):
        predictions = predictions[0]
    if isinstance(predictions, list):
        try:
            predictions = np.stack(predictions, axis=0)
        except Exception:
            predictions = np.concatenate(predictions, axis=0)
    if torch.is_tensor(predictions):
        predictions = predictions.detach().cpu().numpy()
    return np.asarray(predictions)

def safe_roc_auc(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))

def safe_average_precision(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(average_precision_score(y_true, y_prob))

def compute_metrics(eval_pred):
    predictions, labels = eval_pred.predictions, eval_pred.label_ids
    labels = labels.astype(int)

    logits = _extract_logits(predictions)
    if logits.ndim == 3:
        logits = logits[:, 0, :]

    if logits.ndim != 2 or logits.shape[1] < 2:
        raise ValueError(f"Unexpected logits shape: {logits.shape}")

    probs = _softmax_np(logits, axis=-1)[:, 1]

    auc = safe_roc_auc(labels, probs)
    ap = safe_average_precision(labels, probs)
    pred = (probs >= 0.5).astype(int)
    acc = accuracy_score(labels, pred)

    return {
        "auc": float(auc),
        "ap": float(ap),
        "acc@0.5": float(acc),
    }

def probs_from_trainer_predictions(pred_output):
    logits = _extract_logits(pred_output.predictions)
    if logits.ndim == 3:
        logits = logits[:, 0, :]
    probs = _softmax_np(logits, axis=-1)[:, 1]
    labels = pred_output.label_ids.astype(int)
    return labels, probs

def best_threshold_by_f1_grid(y_true, y_prob, step=0.01):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    thresholds = np.arange(0.0, 1.0 + 1e-8, step)

    best_thr = 0.5
    best_f1 = -1.0
    best_prec = float("nan")
    best_rec = float("nan")

    for thr in thresholds:
        pred = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = float(f1)
            best_thr = float(thr)
            best_prec = float(precision_score(y_true, pred, zero_division=0))
            best_rec = float(recall_score(y_true, pred, zero_division=0))

    return {
        "threshold": best_thr,
        "f1": best_f1,
        "precision": best_prec,
        "recall": best_rec,
        "grid_step": float(step),
    }

def save_scores_csv(y_true, y_prob, out_path):
    df = pd.DataFrame({
        "label": np.asarray(y_true).astype(int),
        "prob": np.asarray(y_prob, dtype=float),
    })
    df.to_csv(out_path, index=False)

def save_roc(y_true, p_prob, out_dir, prefix="hold"):
    os.makedirs(out_dir, exist_ok=True)

    fpr, tpr, thresholds = roc_curve(y_true, p_prob)
    auc = safe_roc_auc(y_true, p_prob)

    roc_df = pd.DataFrame({
        "fpr": fpr,
        "tpr": tpr,
        "threshold": thresholds,
    })
    roc_csv = os.path.join(out_dir, f"{prefix}_roc_data.csv")
    roc_df.to_csv(roc_csv, index=False)

    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{prefix.upper()} ROC Curve (DNABERT2)")
    plt.legend(loc="lower right")
    roc_png = os.path.join(out_dir, f"{prefix}_roc_curve.png")
    plt.savefig(roc_png, dpi=200, bbox_inches="tight")
    plt.close()

    return auc, roc_df, roc_csv, roc_png

def mean_roc_from_list(roc_list, fpr_grid=None):
    if fpr_grid is None:
        fpr_grid = np.linspace(0, 1, 200)

    tpr_mat = []

    for df in roc_list:
        df = df.sort_values("fpr").copy()
        fpr = np.clip(df["fpr"].values.astype(float), 0, 1)
        tpr = np.clip(df["tpr"].values.astype(float), 0, 1)

        keep = ~pd.Series(fpr).duplicated().values
        fpr = fpr[keep]
        tpr = tpr[keep]

        if len(fpr) < 2:
            continue

        interp_tpr = np.interp(fpr_grid, fpr, tpr)
        interp_tpr[0] = 0.0
        interp_tpr[-1] = 1.0
        tpr_mat.append(interp_tpr)

    if len(tpr_mat) == 0:
        mean_df = pd.DataFrame({
            "fpr": fpr_grid,
            "mean_tpr": np.nan,
            "std_tpr": np.nan,
        })
        return mean_df, float("nan")

    tpr_mat = np.vstack(tpr_mat)
    mean_tpr = np.nanmean(tpr_mat, axis=0)
    std_tpr = np.nanstd(tpr_mat, axis=0)
    mean_auc = float(np.trapz(mean_tpr, fpr_grid))

    mean_df = pd.DataFrame({
        "fpr": fpr_grid,
        "mean_tpr": mean_tpr,
        "std_tpr": std_tpr,
    })
    return mean_df, mean_auc

# --------------------
# Build HF datasets
# --------------------
def build_hf_ds(df: pd.DataFrame, tokenizer):
    ds = Dataset.from_pandas(df[[SEQ_COL, LABEL_COL]].reset_index(drop=True))

    def tokenize(batch):
        return tokenizer(
            batch[SEQ_COL],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )

    ds = ds.map(tokenize, batched=True)

    if "__index_level_0__" in ds.column_names:
        ds = ds.remove_columns(["__index_level_0__"])

    cols = ["input_ids", "attention_mask", LABEL_COL]
    ds.set_format(type="torch", columns=cols)
    return ds

# --------------------
# One rep run
# --------------------
def run_one_rep(rep_i: int):
    rep_dir = os.path.join(REPS_ROOT, f"rep{rep_i}")
    train_path = os.path.join(rep_dir, "train.csv")
    val_path = os.path.join(rep_dir, "val.csv")
    hold_path = os.path.join(rep_dir, "hold.csv")

    if not (os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(hold_path)):
        print(f"[SKIP] rep{rep_i} missing files.")
        return None

    print(f"\n==================== REP {rep_i} ====================")

    seed_everything(BASE_SEED + rep_i)

    train_df = load_df(train_path)
    val_df = load_df(val_path)
    hold_df = load_df(hold_path)

    print("Train/Val/Hold:", len(train_df), len(val_df), len(hold_df))
    print(
        "pos rate train/val/hold:",
        float(train_df[LABEL_COL].mean()),
        float(val_df[LABEL_COL].mean()),
        float(hold_df[LABEL_COL].mean()),
    )

    if train_df[LABEL_COL].nunique() < 2:
        raise ValueError(f"rep{rep_i}: train set has only one class.")
    if val_df[LABEL_COL].nunique() < 2:
        raise ValueError(f"rep{rep_i}: val set has only one class.")
    if hold_df[LABEL_COL].nunique() < 2:
        raise ValueError(f"rep{rep_i}: hold set has only one class.")

    rep_out = os.path.join(OUT_ROOT, f"rep{rep_i}")
    os.makedirs(rep_out, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
    train_ds = build_hf_ds(train_df, tokenizer)
    val_ds = build_hf_ds(val_df, tokenizer)
    hold_ds = build_hf_ds(hold_df, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR,
        num_labels=2,
        trust_remote_code=True,
    )

    args = TrainingArguments(
        output_dir=rep_out,
        seed=BASE_SEED + rep_i,

        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        num_train_epochs=10,
        weight_decay=0.01,

        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="auc",
        greater_is_better=True,

        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to=[],
        save_total_limit=2,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # --------------------
    # Evaluate best model on VAL
    # --------------------
    val_metrics = trainer.evaluate(val_ds)
    pred_val = trainer.predict(val_ds)
    y_val, p_val = probs_from_trainer_predictions(pred_val)

    val_scores_best_csv = os.path.join(rep_out, "val_scores_best.csv")
    save_scores_csv(y_val, p_val, val_scores_best_csv)

    bt = best_threshold_by_f1_grid(y_val, p_val, step=0.01)
    best_thr = bt["threshold"]

    # --------------------
    # Evaluate best model on HOLD using threshold selected on VAL
    # --------------------
    pred_hold = trainer.predict(hold_ds)
    y_hold, p_hold = probs_from_trainer_predictions(pred_hold)

    hold_scores_best_csv = os.path.join(rep_out, "hold_scores_best.csv")
    save_scores_csv(y_hold, p_hold, hold_scores_best_csv)

    hold_auc = safe_roc_auc(y_hold, p_hold)
    hold_ap = safe_average_precision(y_hold, p_hold)

    pred_hold_label = (p_hold >= best_thr).astype(int)
    hold_acc = float(accuracy_score(y_hold, pred_hold_label))
    hold_prec = float(precision_score(y_hold, pred_hold_label, zero_division=0))
    hold_rec = float(recall_score(y_hold, pred_hold_label, zero_division=0))
    hold_f1 = float(f1_score(y_hold, pred_hold_label, zero_division=0))

    # confusion matrix
    cm = confusion_matrix(y_hold, pred_hold_label, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=["true_0", "true_1"],
        columns=["pred_0", "pred_1"]
    )
    cm_csv = os.path.join(rep_out, "hold_confusion_matrix.csv")
    cm_df.to_csv(cm_csv)

    # roc
    _, roc_df, roc_csv, roc_png = save_roc(y_hold, p_hold, rep_out, prefix="hold")

    # save best model
    best_dir = os.path.join(rep_out, "best_model")
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    # report txt
    report_txt = os.path.join(rep_out, "hold_report.txt")
    with open(report_txt, "w", encoding="utf-8") as f:
        f.write(f"DNABERT2 HOLD RESULTS (rep{rep_i})\n")
        f.write("====================================\n\n")
        f.write("Best model selection metric on VAL: AUC\n")
        f.write(f"VAL AUC (best model): {float(val_metrics.get('eval_auc', float('nan'))):.6f}\n")
        f.write(f"VAL AP  (best model): {float(val_metrics.get('eval_ap', float('nan'))):.6f}\n")
        f.write(f"VAL ACC@0.5          : {float(val_metrics.get('eval_acc@0.5', float('nan'))):.6f}\n\n")

        f.write("Threshold selected ONLY on VAL by best F1\n")
        f.write("Threshold search grid: 0.00 to 1.00 with step 0.01\n")
        f.write(f"Best threshold: {best_thr:.6f}\n")
        f.write(f"VAL best F1   : {bt['f1']:.6f}\n")
        f.write(f"VAL precision : {bt['precision']:.6f}\n")
        f.write(f"VAL recall    : {bt['recall']:.6f}\n\n")

        f.write("HOLD metrics\n")
        f.write(f"HOLD AUC      : {hold_auc:.6f}\n")
        f.write(f"HOLD AP       : {hold_ap:.6f}\n")
        f.write(f"HOLD ACC      : {hold_acc:.6f}\n")
        f.write(f"HOLD Precision: {hold_prec:.6f}\n")
        f.write(f"HOLD Recall   : {hold_rec:.6f}\n")
        f.write(f"HOLD F1       : {hold_f1:.6f}\n\n")

        f.write("Confusion Matrix (rows=true, cols=pred)\n")
        f.write(cm_df.to_string())
        f.write("\n")

    summary = {
        "rep": rep_i,
        "best_model_dir": best_dir,
        "model_selection": {
            "metric": "val_auc",
            "threshold_selection_metric": "val_f1",
            "threshold_search_grid": {
                "start": 0.0,
                "end": 1.0,
                "step": 0.01,
            },
        },
        "val_metrics_best_model": {
            "auc": float(val_metrics.get("eval_auc", float("nan"))),
            "ap": float(val_metrics.get("eval_ap", float("nan"))),
            "acc@0.5": float(val_metrics.get("eval_acc@0.5", float("nan"))),
        },
        "val_best_threshold": {
            "threshold": float(best_thr),
            "f1": float(bt["f1"]),
            "precision": float(bt["precision"]),
            "recall": float(bt["recall"]),
        },
        "hold_metrics": {
            "auc": float(hold_auc),
            "ap": float(hold_ap),
            "acc": float(hold_acc),
            "precision": float(hold_prec),
            "recall": float(hold_rec),
            "f1": float(hold_f1),
        },
        "artifacts": {
            "best_model_dir": best_dir,
            "val_scores_best_csv": val_scores_best_csv,
            "hold_scores_best_csv": hold_scores_best_csv,
            "hold_roc_curve_png": roc_png,
            "hold_roc_data_csv": roc_csv,
            "hold_confusion_matrix_csv": cm_csv,
            "hold_report_txt": report_txt,
        }
    }

    summary_json = os.path.join(rep_out, "hold_summary.json")
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(
        f"VAL : AUC={summary['val_metrics_best_model']['auc']:.4f} "
        f"AP={summary['val_metrics_best_model']['ap']:.4f} "
        f"ACC@0.5={summary['val_metrics_best_model']['acc@0.5']:.4f}"
    )
    print(
        f"VAL threshold by best F1: thr={best_thr:.2f} "
        f"F1={bt['f1']:.4f} P={bt['precision']:.4f} R={bt['recall']:.4f}"
    )
    print(
        f"HOLD: AUC={hold_auc:.4f} AP={hold_ap:.4f} "
        f"ACC={hold_acc:.4f} P={hold_prec:.4f} R={hold_rec:.4f} F1={hold_f1:.4f}"
    )
    print("Saved to:", os.path.abspath(rep_out))

    return {
        "rep": rep_i,
        "val_auc_best_model": float(summary["val_metrics_best_model"]["auc"]),
        "val_ap_best_model": float(summary["val_metrics_best_model"]["ap"]),
        "val_acc@0.5_best_model": float(summary["val_metrics_best_model"]["acc@0.5"]),
        "threshold_from_val_f1": float(best_thr),
        "val_best_f1": float(bt["f1"]),
        "val_best_precision": float(bt["precision"]),
        "val_best_recall": float(bt["recall"]),
        "hold_auc": float(hold_auc),
        "hold_ap": float(hold_ap),
        "hold_acc": float(hold_acc),
        "hold_precision": float(hold_prec),
        "hold_recall": float(hold_rec),
        "hold_f1": float(hold_f1),
        "best_model_dir": best_dir,
        "hold_roc_curve_png": roc_png,
        "hold_roc_data_csv": roc_csv,
        "hold_confusion_matrix_csv": cm_csv,
        "hold_report_txt": report_txt,
        "hold_summary_json": summary_json,
        "pos_rate_hold": float(np.mean(y_hold == 1)),
        "n_hold": int(len(y_hold)),
        "roc_df": roc_df,
    }

# --------------------
# Main: loop over reps + global summaries
# --------------------
def main():
    rows = []
    roc_list = []
    auc_rows = []

    for rep_i in range(1, N_REP + 1):
        out = run_one_rep(rep_i)
        if out is not None:
            rows.append({
                "rep": out["rep"],
                "val_auc_best_model": out["val_auc_best_model"],
                "val_ap_best_model": out["val_ap_best_model"],
                "val_acc@0.5_best_model": out["val_acc@0.5_best_model"],
                "threshold_from_val_f1": out["threshold_from_val_f1"],
                "val_best_f1": out["val_best_f1"],
                "val_best_precision": out["val_best_precision"],
                "val_best_recall": out["val_best_recall"],
                "hold_auc": out["hold_auc"],
                "hold_ap": out["hold_ap"],
                "hold_acc": out["hold_acc"],
                "hold_precision": out["hold_precision"],
                "hold_recall": out["hold_recall"],
                "hold_f1": out["hold_f1"],
                "n_hold": out["n_hold"],
                "pos_rate_hold": out["pos_rate_hold"],
                "best_model_dir": out["best_model_dir"],
                "hold_roc_curve_png": out["hold_roc_curve_png"],
                "hold_roc_data_csv": out["hold_roc_data_csv"],
                "hold_confusion_matrix_csv": out["hold_confusion_matrix_csv"],
                "hold_report_txt": out["hold_report_txt"],
                "hold_summary_json": out["hold_summary_json"],
            })

            rep_roc_df = out["roc_df"].copy()
            rep_roc_df["rep"] = out["rep"]
            roc_list.append(rep_roc_df)

            auc_rows.append({
                "rep": out["rep"],
                "hold_auc": out["hold_auc"],
            })

    if not rows:
        print("No reps processed.")
        return

    # all reps summary
    summary_df = pd.DataFrame(rows).sort_values("rep")
    summary_csv = os.path.join(OUT_ROOT, "all_reps_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    # all reps roc summary
    if len(roc_list) > 0:
        roc_long = pd.concat(roc_list, axis=0, ignore_index=True)
        roc_long = roc_long.sort_values(["rep", "fpr"])
    else:
        roc_long = pd.DataFrame(columns=["fpr", "tpr", "threshold", "rep"])
    roc_long_csv = os.path.join(OUT_ROOT, "all_reps_roc_summary.csv")
    roc_long.to_csv(roc_long_csv, index=False)

    # mean roc curve
    mean_roc_df, mean_auc = mean_roc_from_list(roc_list)
    mean_roc_csv = os.path.join(OUT_ROOT, "mean_roc_curve.csv")
    mean_roc_df.to_csv(mean_roc_csv, index=False)

    # auc summary
    auc_df = pd.DataFrame(auc_rows).sort_values("rep")
    auc_csv = os.path.join(OUT_ROOT, "all_reps_auc_summary.csv")
    auc_df.to_csv(auc_csv, index=False)

    print("\n==================== ALL REPS SUMMARY ====================")
    print(summary_df)
    print(f"\nSaved: {os.path.abspath(summary_csv)}")
    print(f"Saved: {os.path.abspath(roc_long_csv)}")
    print(f"Saved: {os.path.abspath(mean_roc_csv)}")
    print(f"Saved: {os.path.abspath(auc_csv)}")
    print(f"Mean AUC (interpolated): {mean_auc:.4f}")

if __name__ == "__main__":
    main()


/root/miniconda3/envs/DNABERT2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/DNABERT2/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



==================== REP 1 ====================
Train/Val/Hold: 14152 3532 4411
pos rate train/val/hold: 0.5000706613906162 0.5 0.4996599410564498


Map: 100%|██████████| 4411/4411 [00:00<00:00, 7243.66 examples/s]
/root/.cache/huggingface/modules/transformers_modules/model/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of the model checkpoint at model were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkp

Epoch,Training Loss,Validation Loss,Auc,Ap,Acc@0.5
1,0.509300,0.483078,0.851729,0.847954,0.772084
2,0.419000,0.467388,0.882937,0.878783,0.797565
3,0.343300,0.495724,0.884267,0.876681,0.795583
4,0.226500,0.664165,0.873184,0.865561,0.799830
5,0.220600,0.699931,0.861884,0.843758,0.801246
6,0.124300,0.892843,0.866999,0.850392,0.795583
7,0.061400,1.004743,0.860315,0.839390,0.801246
8,0.058100,0.957168,0.840779,0.795977,0.799547
9,0.060000,1.103313,0.850325,0.811341,0.799547
10,0.033400,1.153843,0.851774,0.818197,0.802945


VAL : AUC=0.8843 AP=0.8767 ACC@0.5=0.7956
VAL threshold by best F1: thr=0.53 F1=0.8154 P=0.7519 R=0.8907
HOLD: AUC=0.8678 AP=0.8570 ACC=0.7844 P=0.7407 R=0.8748 F1=0.8022
Saved to: /root/autodl-tmp/DNABERT_2-main/dnabert2_by_rep_valauc_valf1_grid001/rep1

==================== REP 2 ====================
Train/Val/Hold: 14141 3535 4419
pos rate train/val/hold: 0.5002475072484266 0.4995756718528996 0.49943426114505546


...

Epoch,Training Loss,Validation Loss,Auc,Ap,Acc@0.5
1,0.548500,0.546747,0.808327,0.795648,0.717237
2,0.457600,0.511341,0.859060,0.850285,0.778658
3,0.355100,0.524346,0.857179,0.851318,0.784602
4,0.268200,0.675942,0.858696,0.853504,0.769884
5,0.146600,0.871340,0.854505,0.839716,0.766770
6,0.142100,0.998193,0.848984,0.839950,0.778941
7,0.080700,1.163804,0.839777,0.807539,0.774130
8,0.091400,1.196080,0.837349,0.802263,0.774979
9,0.098900,1.280123,0.815665,0.763236,0.771016
10,0.031600,1.329546,0.821064,0.772674,0.770733


VAL : AUC=0.8591 AP=0.8503 ACC@0.5=0.7787
VAL threshold by best F1: thr=0.29 F1=0.7882 P=0.7251 R=0.8633
HOLD: AUC=0.8608 AP=0.8499 ACC=0.7771 P=0.7308 R=0.8773 F1=0.7974
Saved to: /root/autodl-tmp/DNABERT_2-main/dnabert2_by_rep_valauc_valf1_grid001/rep9

==================== REP 10 ====================
Train/Val/Hold: 14147 3533 4415
pos rate train/val/hold: 0.4996819113592988 0.49985847721483156 0.5010192525481314


Map: 100%|██████████| 4415/4415 [00:00<00:00, 9054.02 examples/s]
/root/.cache/huggingface/modules/transformers_modules/model/bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of the model checkpoint at model were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkp

Epoch,Training Loss,Validation Loss,Auc,Ap,Acc@0.5
1,0.519000,0.531201,0.815326,0.811388,0.738466
2,0.436100,0.506660,0.864221,0.855415,0.776394
3,0.334300,0.550255,0.862177,0.855109,0.781772
4,0.243400,0.633917,0.860029,0.849434,0.771865
5,0.186300,0.849948,0.847721,0.822562,0.774979
6,0.159900,0.866482,0.854671,0.839616,0.766487
7,0.080800,1.052961,0.847056,0.824135,0.774130
8,0.087400,1.132248,0.839363,0.812247,0.772148
9,0.090100,1.174115,0.833907,0.799223,0.772714
10,0.049300,1.214461,0.824102,0.780219,0.775545


VAL : AUC=0.8642 AP=0.8554 ACC@0.5=0.7764
VAL threshold by best F1: thr=0.47 F1=0.7949 P=0.7316 R=0.8703
HOLD: AUC=0.8583 AP=0.8424 ACC=0.7780 P=0.7391 R=0.8608 F1=0.7953
Saved to: /root/autodl-tmp/DNABERT_2-main/dnabert2_by_rep_valauc_valf1_grid001/rep10

==================== ALL REPS SUMMARY ====================
   rep  val_auc_best_model  val_ap_best_model  val_acc@0.5_best_model  \
0    1            0.884267           0.876681                0.795583   
1    2            0.869279           0.866852                0.782461   
2    3            0.856214           0.850423                0.769731   
3    4            0.878582           0.870978                0.806561   
4    5            0.870053           0.860459                0.777086   
5    6            0.859352           0.848216                0.776111   
6    7            0.849881           0.843226                0.758357   
7    8            0.864972           0.862873                0.772213   
8    9            0.859060 